# Feast offline-to-online training and inference

This notebook runs a production-shaped, CPU-only feature pipeline:

1. a Spark Operator `SparkApplication` generates a synthetic batch dataset and stores it as Parquet in S3-compatible object storage;
2. Spark performs Feast's point-in-time historical join and trains a small NumPy model;
3. Feast uses PostgreSQL for its durable SQL registry and materializes the latest feature values into Redis;
4. KServe deploys a model server that reads Redis-backed online features before predicting.

This separation is the recommended starting point for larger workloads: object storage is the durable, versionable offline data layer; operator-managed Spark provides elastic batch compute; PostgreSQL stores Feast metadata rather than the feature history; and Redis serves low-latency online lookups. Feast classifies its Spark offline store as a contributed integration without full test coverage, so qualify it against your scale and upgrade requirements or use a fully supported warehouse while retaining the same S3-and-Spark data pipeline. For production, use highly available PostgreSQL, Redis, and S3-compatible services with TLS, secret rotation, backups, monitoring, retention policies, and scheduled `SparkApplication` materialization runs.

Prerequisites:

- the Alauda Spark Operator is installed through OLM and `sparkapplications.sparkoperator.k8s.io` exists;
- a Spark runtime image containing PySpark, Feast 0.61.x with Spark and Redis support, NumPy, pandas, PyArrow, PyYAML, and a Hadoop S3A connector compatible with the image's Hadoop version;
- a `feast-data-stores` Secret with `redis` and `sql` keys for the Feast Operator;
- a `feast-s3-credentials` Secret with `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_ENDPOINT_URL`, and `S3_BUCKET`;
- a pre-created S3 bucket and a ReadWriteOnce storage class for the sample model PVC; and
- the Feast and KServe operators.

Do not copy an internal registry hostname from this document. Find the registry for your global cluster with:

```bash
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
```

In the ACP console, open the container registry associated with that address, choose an approved Spark runtime that satisfies the dependency list above, and set its complete reference in `FEAST_SPARK_IMAGE`. Set `FEAST_MODEL_IMAGE` similarly if the default Feast model-server repository or tag is not mirrored in your global registry.


In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

import requests

NAMESPACE = os.environ.get("FEAST_NAMESPACE", "feast-demo")
FEATURESTORE_NAME = os.environ.get("FEAST_FEATURESTORE", "feast-notebook")
FEAST_PROJECT = os.environ.get("FEAST_PROJECT", "feast_demo")
MODEL_PVC = os.environ.get("FEAST_MODEL_PVC", "feast-notebook-model")
MODEL_RUNTIME = os.environ.get("FEAST_MODEL_RUNTIME", "feast-numpy-runtime")
MODEL_NAME = os.environ.get("FEAST_MODEL_NAME", "feast-online-model")
SPARK_APP = os.environ.get("FEAST_SPARK_APPLICATION", "feast-offline-batch")
SPARK_SERVICE_ACCOUNT = os.environ.get("FEAST_SPARK_SERVICE_ACCOUNT", "feast-spark")
SPARK_VERSION = os.environ.get("FEAST_SPARK_VERSION", "4.0.1")
DATA_STORES_SECRET = os.environ.get("FEAST_DATA_STORES_SECRET", "feast-data-stores")
S3_CREDENTIALS_SECRET = os.environ.get("FEAST_S3_CREDENTIALS_SECRET", "feast-s3-credentials")
S3_DATASET_KEY = os.environ.get("FEAST_S3_DATASET_KEY", "datasets/driver_stats")
def kubectl(*args, input_text=None, check=True):
    result = subprocess.run(["kubectl", *args], input=input_text, text=True, capture_output=True)
    if check and result.returncode:
        raise RuntimeError(f"kubectl {' '.join(args)} failed: {result.stderr}")
    return result.stdout.strip()

def secret_exists(name):
    return bool(kubectl("get", "secret", name, "-n", NAMESPACE, "-o", "name", check=False))

registry_address = kubectl(
    "get", "configmap", "global-info", "-n", "kube-public",
    "-o", "jsonpath={.data.registryAddress}",
)
if not registry_address:
    raise RuntimeError("kube-public/global-info does not contain data.registryAddress")

SPARK_IMAGE = os.environ.get("FEAST_SPARK_IMAGE")
if not SPARK_IMAGE:
    raise RuntimeError(
        "Set FEAST_SPARK_IMAGE to an approved Spark+Feast runtime from the registry "
        f"reported by kube-public/global-info ({registry_address})."
    )
MODEL_IMAGE = os.environ.get(
    "FEAST_MODEL_IMAGE", f"{registry_address}/mlops/feast/feature-server:0.61.0"
)

if not kubectl("get", "crd", "sparkapplications.sparkoperator.k8s.io", "-o", "name", check=False):
    raise RuntimeError("Install the OLM Spark Operator before continuing")

print({
    "namespace": NAMESPACE,
    "featurestore": FEATURESTORE_NAME,
    "spark_application": SPARK_APP,
    "spark_registry": registry_address,
})


## 1. Prepare PostgreSQL registry and Redis online serving

The Feast Operator manages only the control plane and online-serving backends here. PostgreSQL stores the Feast SQL registry; Redis stores materialized online feature values. The Spark driver overrides the generated client configuration with Feast's Spark offline store, so no Feast offline-server pod or separate Spark/Hadoop installation is needed.


In [ ]:
for namespace in (NAMESPACE, "feast-operator-system"):
    namespace_yaml = kubectl("create", "namespace", namespace, "--dry-run=client", "-o", "yaml")
    kubectl("apply", "-f", "-", input_text=namespace_yaml)

for secret_name in (DATA_STORES_SECRET, S3_CREDENTIALS_SECRET):
    if not secret_exists(secret_name):
        raise RuntimeError(f"Create Secret {NAMESPACE}/{secret_name} before continuing")

featurestore_yaml = f"""
apiVersion: feast.dev/v1
kind: FeatureStore
metadata:
  name: {FEATURESTORE_NAME}
  namespace: {NAMESPACE}
spec:
  feastProject: {FEAST_PROJECT}
  replicas: 1
  services:
    onlineStore:
      persistence:
        store:
          type: redis
          secretRef:
            name: {DATA_STORES_SECRET}
    registry:
      local:
        persistence:
          store:
            type: sql
            secretRef:
              name: {DATA_STORES_SECRET}
        server: {{}}
    ui: {{}}
"""
kubectl("apply", "-f", "-", input_text=featurestore_yaml)

deadline = time.time() + 600
while time.time() < deadline:
    phase = kubectl(
        "get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE,
        "-o", "jsonpath={.status.phase}", check=False,
    )
    print(phase or "Pending")
    if phase == "Ready":
        break
    if phase == "Failed":
        raise RuntimeError(kubectl("describe", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, check=False))
    time.sleep(10)
else:
    raise TimeoutError("FeatureStore did not become Ready")

client_config_map = kubectl(
    "get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE,
    "-o", "jsonpath={.status.clientConfigMap}",
)
print("FeatureStore is Ready; client ConfigMap:", client_config_map)


## 2. Prepare the synthetic S3 batch application

This section creates the code that the Spark Operator will mount into the driver. The application generates deterministic synthetic driver events with Spark, writes partitioned Parquet to the configured S3-compatible service, registers a Feast `SparkSource`, performs the point-in-time historical join, trains the sample model, and materializes the same features into Redis.

The S3 endpoint and credentials come from a Kubernetes Secret. The code enables path-style access so it works with SeaweedFS and similar S3-compatible services; use your service's TLS endpoint in production.


In [ ]:
batch_source = r'''import copy
import json
import os
import shutil
import subprocess
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import yaml
from feast import FeatureStore
from pyspark.sql import SparkSession, functions as F

project = os.environ["FEAST_PROJECT"]
bucket = os.environ["S3_BUCKET"]
dataset_key = os.environ["S3_DATASET_KEY"].strip("/")
dataset_uri = f"s3a://{bucket}/{dataset_key}"
region = os.environ["AWS_DEFAULT_REGION"]
repo = Path("/tmp/feast-repo")
model_repo = Path("/mnt/models/repo")
repo.mkdir(parents=True, exist_ok=True)
model_repo.mkdir(parents=True, exist_ok=True)

spark = SparkSession.builder.appName("feast-offline-batch").getOrCreate()
endpoint = urlparse(os.environ["S3_ENDPOINT_URL"])
hadoop = spark.sparkContext._jsc.hadoopConfiguration()
hadoop.set("fs.s3a.endpoint", endpoint.netloc or endpoint.path)
hadoop.set("fs.s3a.endpoint.region", region)
hadoop.set("fs.s3a.path.style.access", "true")
hadoop.set("fs.s3a.connection.ssl.enabled", str(endpoint.scheme == "https").lower())
hadoop.set("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
hadoop.set("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])

n_rows = 240
events = (
    spark.range(n_rows)
    .withColumn("driver_id", (F.col("id") % 12 + 1).cast("long"))
    .withColumn("event_timestamp", F.timestamp_seconds(F.lit(1767225600) + F.col("id") * 3600))
    .withColumn("created", F.col("event_timestamp") + F.expr("INTERVAL 1 MINUTE"))
    .withColumn("conv_rate", (F.lit(0.25) + F.lit(0.55) * F.rand(7)).cast("float"))
    .withColumn("acc_rate", (F.lit(0.50) + F.lit(0.45) * F.rand(11)).cast("float"))
    .withColumn("avg_daily_trips", F.floor(F.lit(2) + F.lit(18) * F.rand(13)).cast("long"))
    .withColumn(
        "label",
        ((F.col("conv_rate") * 2 + F.col("acc_rate") + F.col("avg_daily_trips") / 20) > 1.8).cast("long"),
    )
    .drop("id")
)
events.repartition(4, "driver_id").write.mode("overwrite").partitionBy("driver_id").parquet(dataset_uri)

client_config = yaml.safe_load(Path("/etc/feast/feature_store.yaml").read_text())
batch_config = copy.deepcopy(client_config)
batch_config["offline_store"] = {
    "type": "spark",
    "spark_conf": {
        "spark.sql.session.timeZone": "UTC",
        "spark.hadoop.fs.s3a.endpoint": endpoint.netloc or endpoint.path,
        "spark.hadoop.fs.s3a.endpoint.region": region,
        "spark.hadoop.fs.s3a.path.style.access": "true",
        "spark.hadoop.fs.s3a.connection.ssl.enabled": str(endpoint.scheme == "https").lower(),
    },
}
(repo / "feature_store.yaml").write_text(yaml.safe_dump(batch_config, sort_keys=False))
(repo / "features.py").write_text(f"""from datetime import timedelta
from feast import Entity, FeatureService, FeatureView, Field
from feast.infra.offline_stores.contrib.spark_offline_store.spark_source import SparkSource
from feast.types import Float32, Int64
from feast.value_type import ValueType

driver = Entity(name="driver", join_keys=["driver_id"], value_type=ValueType.INT64)
source = SparkSource(
    name="driver_stats_source",
    path="{dataset_uri}",
    file_format="parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)
view = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=365),
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    online=True,
    source=source,
)
driver_activity_v1 = FeatureService(name="driver_activity_v1", features=[view])
""")

subprocess.run(["feast", "--chdir", str(repo), "apply"], check=True)
store = FeatureStore(repo_path=str(repo))
entity_df = events.select("driver_id", "event_timestamp", "label").toPandas()
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df().dropna()
columns = ["conv_rate", "acc_rate", "avg_daily_trips"]
x = training_df[columns].to_numpy(dtype="float64")
y = training_df["label"].to_numpy(dtype="float64")
weights = np.linalg.pinv(np.column_stack([np.ones(len(x)), x])) @ y
np.savez("/mnt/models/model.npz", weights=weights, feature_columns=np.array(columns))

store.materialize_incremental(events.agg(F.max("event_timestamp")).first()[0] + pd.Timedelta(hours=1))
online = store.get_online_features(
    features=store.get_feature_service("driver_activity_v1"),
    entity_rows=[{"driver_id": 1}, {"driver_id": 2}],
).to_df()
online.to_json("/mnt/models/online-sample.json", orient="records")
serving_config = copy.deepcopy(client_config)
serving_config.pop("offline_store", None)
(model_repo / "feature_store.yaml").write_text(yaml.safe_dump(serving_config, sort_keys=False))
shutil.copy("/opt/feast-batch/server.py", model_repo / "server.py")
print(json.dumps({"rows": len(training_df), "dataset": dataset_uri, "online_rows": len(online)}))
spark.stop()
'''

server_source = r'''import os
import numpy as np
from fastapi import Body, FastAPI
from feast import FeatureStore
import uvicorn

MODEL_NAME = os.getenv("MODEL_NAME", "feast-online-model")
weights = np.load("/mnt/models/model.npz")["weights"]
store = FeatureStore(repo_path="/mnt/models/repo")
feature_service = store.get_feature_service("driver_activity_v1")
app = FastAPI()

@app.get("/v2/health/live")
@app.get("/v2/health/ready")
def ready():
    return {"ready": True}

@app.get("/v2/models/{model_name}")
@app.get("/v2/models/{model_name}/ready")
def model_ready(model_name: str):
    return {"name": model_name, "ready": model_name == MODEL_NAME}

@app.post("/v2/models/{model_name}/infer")
def infer(model_name: str, payload: dict = Body(...)):
    ids = next(item for item in payload["inputs"] if item["name"] == "driver_id")["data"]
    rows = [{"driver_id": int(driver_id)} for driver_id in ids]
    values = store.get_online_features(features=feature_service, entity_rows=rows).to_dict()
    def column(name):
        if name in values:
            return values[name]
        return values[next(key for key in values if key.endswith("__" + name))]
    x = np.column_stack([np.ones(len(ids)), column("conv_rate"), column("acc_rate"), column("avg_daily_trips")])
    prediction = (x @ weights).astype("float32")
    return {"model_name": model_name, "outputs": [{"name": "prediction", "shape": [len(ids)], "datatype": "FP32", "data": prediction.tolist()}]}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080)
'''

batch_dir = Path("feast-spark-batch")
batch_dir.mkdir(exist_ok=True)
(batch_dir / "batch.py").write_text(batch_source)
(batch_dir / "server.py").write_text(server_source)
print("Prepared", batch_dir)


## 3. Submit the batch work as a SparkApplication

The Spark Operator creates and monitors the driver and executor pods from this CR. A namespace-scoped service account gives the driver only the permissions it needs to manage its executors. The model PVC is mounted only in the driver; the dataset remains in S3.


In [ ]:
rbac_yaml = f"""
apiVersion: v1
kind: ServiceAccount
metadata:
  name: {SPARK_SERVICE_ACCOUNT}
  namespace: {NAMESPACE}
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: {SPARK_SERVICE_ACCOUNT}
  namespace: {NAMESPACE}
rules:
- apiGroups: [""]
  resources: ["pods", "pods/log", "services", "configmaps"]
  verbs: ["get", "list", "watch", "create", "delete", "patch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: {SPARK_SERVICE_ACCOUNT}
  namespace: {NAMESPACE}
subjects:
- kind: ServiceAccount
  name: {SPARK_SERVICE_ACCOUNT}
  namespace: {NAMESPACE}
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: {SPARK_SERVICE_ACCOUNT}
"""
kubectl("apply", "-f", "-", input_text=rbac_yaml)

pvc_yaml = f"""
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: {MODEL_PVC}
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 1Gi
"""
kubectl("apply", "-f", "-", input_text=pvc_yaml)

batch_config_map = f"{SPARK_APP}-code"
config_map_yaml = kubectl(
    "create", "configmap", batch_config_map, "-n", NAMESPACE,
    f"--from-file=batch.py={Path('feast-spark-batch/batch.py')}",
    f"--from-file=server.py={Path('feast-spark-batch/server.py')}",
    "--dry-run=client", "-o", "yaml",
)
kubectl("apply", "-f", "-", input_text=config_map_yaml)

spark_application_yaml = f"""
apiVersion: sparkoperator.k8s.io/v1beta2
kind: SparkApplication
metadata:
  name: {SPARK_APP}
  namespace: {NAMESPACE}
spec:
  type: Python
  mode: cluster
  image: {SPARK_IMAGE}
  imagePullPolicy: IfNotPresent
  mainApplicationFile: local:///opt/feast-batch/batch.py
  sparkVersion: {SPARK_VERSION}
  timeToLiveSeconds: 3600
  restartPolicy:
    type: Never
  sparkConf:
    spark.sql.session.timeZone: UTC
  volumes:
  - name: batch-code
    configMap:
      name: {batch_config_map}
  - name: feast-client
    configMap:
      name: {client_config_map}
      items:
      - key: feature_store.yaml
        path: feature_store.yaml
  - name: model
    persistentVolumeClaim:
      claimName: {MODEL_PVC}
  - name: online-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-online-tls
  - name: registry-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-registry-tls
  driver:
    cores: 1
    memory: 2g
    serviceAccount: {SPARK_SERVICE_ACCOUNT}
    env:
    - name: FEAST_PROJECT
      value: {FEAST_PROJECT}
    - name: S3_DATASET_KEY
      value: {S3_DATASET_KEY}
    envFrom:
    - secretRef:
        name: {S3_CREDENTIALS_SECRET}
    volumeMounts:
    - name: batch-code
      mountPath: /opt/feast-batch
      readOnly: true
    - name: feast-client
      mountPath: /etc/feast
      readOnly: true
    - name: model
      mountPath: /mnt/models
    - name: online-tls
      mountPath: /tls/online
      readOnly: true
    - name: registry-tls
      mountPath: /tls/registry
      readOnly: true
  executor:
    instances: 2
    cores: 1
    memory: 1g
    envFrom:
    - secretRef:
        name: {S3_CREDENTIALS_SECRET}
"""
kubectl("delete", "sparkapplication", SPARK_APP, "-n", NAMESPACE, "--ignore-not-found", "--wait=true")
kubectl("apply", "-f", "-", input_text=spark_application_yaml)

deadline = time.time() + 1200
terminal = {"COMPLETED", "FAILED", "FAILED_SUBMISSION", "INVALIDATING", "UNKNOWN"}
while time.time() < deadline:
    state = kubectl(
        "get", "sparkapplication", SPARK_APP, "-n", NAMESPACE,
        "-o", "jsonpath={.status.applicationState.state}", check=False,
    )
    print(state or "SUBMITTED")
    if state in terminal:
        break
    time.sleep(10)
else:
    raise TimeoutError("SparkApplication did not reach a terminal state")

if state != "COMPLETED":
    driver_pod = kubectl(
        "get", "sparkapplication", SPARK_APP, "-n", NAMESPACE,
        "-o", "jsonpath={.status.driverInfo.podName}", check=False,
    )
    logs = kubectl("logs", driver_pod, "-n", NAMESPACE, "--tail=300", check=False) if driver_pod else ""
    raise RuntimeError(f"SparkApplication ended in {state}\n{logs}")

print("Spark batch completed")


## 4. Inspect the offline-to-online result

The driver writes a small verification sample to the model PVC after materialization. Inspecting it through a short-lived pod confirms that the Spark batch completed the S3 historical path and that Feast could read the materialized Redis values.


In [ ]:
inspector_yaml = f"""
apiVersion: v1
kind: Pod
metadata:
  name: feast-model-inspector
  namespace: {NAMESPACE}
spec:
  restartPolicy: Never
  containers:
  - name: inspector
    image: {MODEL_IMAGE}
    command: [bash, -c, cat /mnt/models/online-sample.json]
    volumeMounts:
    - name: model
      mountPath: /mnt/models
  volumes:
  - name: model
    persistentVolumeClaim:
      claimName: {MODEL_PVC}
"""
kubectl("delete", "pod", "feast-model-inspector", "-n", NAMESPACE, "--ignore-not-found", "--wait=true")
kubectl("apply", "-f", "-", input_text=inspector_yaml)
kubectl("wait", "--for=jsonpath={.status.phase}=Succeeded", "pod/feast-model-inspector", "-n", NAMESPACE, "--timeout=180s")
print(kubectl("logs", "feast-model-inspector", "-n", NAMESPACE))
kubectl("delete", "pod", "feast-model-inspector", "-n", NAMESPACE, "--wait=true")


## 5. Deploy the online model with KServe

The model server loads the weights and Feast definitions from the PVC, queries Redis through the operator-managed Feast online service, and exposes KServe's v2 inference protocol. The online and registry TLS Secrets are mounted at the paths in the generated Feast client configuration.


In [ ]:
runtime_yaml = f"""
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  name: {MODEL_RUNTIME}
  namespace: {NAMESPACE}
spec:
  containers:
  - name: kserve-container
    image: {MODEL_IMAGE}
    command: [python, /mnt/models/repo/server.py]
    ports:
    - containerPort: 8080
      name: http1
      protocol: TCP
    env:
    - name: MODEL_NAME
      value: {MODEL_NAME}
    volumeMounts:
    - name: online-tls
      mountPath: /tls/online
      readOnly: true
    - name: registry-tls
      mountPath: /tls/registry
      readOnly: true
  protocolVersions: [v2]
  supportedModelFormats:
  - name: feast-numpy
    version: "1"
  volumes:
  - name: online-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-online-tls
  - name: registry-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-registry-tls
"""
isvc_yaml = f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {MODEL_NAME}
  namespace: {NAMESPACE}
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: feast-numpy
        version: "1"
      protocolVersion: v2
      runtime: {MODEL_RUNTIME}
      storageUri: pvc://{MODEL_PVC}
      resources:
        requests:
          cpu: "100m"
          memory: 256Mi
        limits:
          cpu: "1"
          memory: 1Gi
"""
kubectl("apply", "-f", "-", input_text=runtime_yaml)
kubectl("apply", "-f", "-", input_text=isvc_yaml)
print(kubectl("get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE))


## 6. Send an online-feature prediction

After the predictor has an available replica, send driver IDs to the KServe v2 endpoint. The server looks up their materialized features in Redis and combines those values with the model trained by the `SparkApplication`.


In [ ]:
deadline = time.time() + 900
predictor_deployment = f"{MODEL_NAME}-predictor"
while time.time() < deadline:
    deployment_text = kubectl(
        "get", "deployment", predictor_deployment, "-n", NAMESPACE, "-o", "json", check=False
    )
    deployment = json.loads(deployment_text) if deployment_text else {}
    available = deployment.get("status", {}).get("availableReplicas", 0) or 0
    print({"availableReplicas": available})
    if available >= 1:
        break
    time.sleep(10)
else:
    raise TimeoutError("KServe predictor deployment did not become available")

port_forward = subprocess.Popen(
    ["kubectl", "port-forward", f"service/{predictor_deployment}", "18080:80", "-n", NAMESPACE],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
base_url = "http://127.0.0.1:18080"
try:
    deadline = time.time() + 60
    while time.time() < deadline:
        if port_forward.poll() is not None:
            raise RuntimeError("kubectl port-forward exited unexpectedly")
        try:
            if requests.get(f"{base_url}/v2/health/ready", timeout=2).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise TimeoutError("KServe predictor endpoint did not become ready")

    response = requests.post(
        f"{base_url}/v2/models/{MODEL_NAME}/infer",
        json={"inputs": [{"name": "driver_id", "shape": [2], "datatype": "INT64", "data": [1, 2]}]},
        timeout=30,
    )
    response.raise_for_status()
    print(json.dumps(response.json(), indent=2))
finally:
    port_forward.terminate()
    try:
        port_forward.wait(timeout=5)
    except subprocess.TimeoutExpired:
        port_forward.kill()
        port_forward.wait()
